In [20]:
import torch
print(torch.__version__)

2.10.0+cu128


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 先定好超参数，后面一直用这组数字
B = 2      # batch size
T = 5      # sequence length（token数）
d_model = 8  # 每个token的embedding维度（论文里是512，我们用小数方便打印）

# 造一个假输入
x = torch.randn(B, T, d_model)
print(x.shape)  # 期望: torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


In [22]:
d_k = d_model  # 单头情况下，d_k = d_model

# 三个线性变换，把x投影成Q、K、V
W_q = nn.Linear(d_model, d_k, bias=False)
W_k = nn.Linear(d_model, d_k, bias=False)
W_v = nn.Linear(d_model, d_k, bias=False)

Q = W_q(x)  # (B, T, d_k)
K = W_k(x)  # (B, T, d_k)
V = W_v(x)  # (B, T, d_k)

print(Q.shape, K.shape, V.shape)
# 期望: torch.Size([2, 5, 8]) x3

torch.Size([2, 5, 8]) torch.Size([2, 5, 8]) torch.Size([2, 5, 8])


In [23]:
# Q和K做点积，得到每对token之间的相关性分数
scores = Q @ K.transpose(-2, -1)  # (B, T, T)
print(scores.shape)  # 期望: torch.Size([2, 5, 5])

torch.Size([2, 5, 5])


In [24]:
scores = scores / math.sqrt(d_k)
attn_weights = F.softmax(scores, dim=-1)  # (B, T, T)
print(attn_weights.shape)
print(attn_weights[0])  # 打印第一个样本的注意力矩阵，看看每行是否加和为1

torch.Size([2, 5, 5])
tensor([[0.1210, 0.1787, 0.2681, 0.1190, 0.3132],
        [0.2718, 0.2309, 0.1512, 0.1729, 0.1732],
        [0.2346, 0.1705, 0.1537, 0.3104, 0.1309],
        [0.0983, 0.2026, 0.2477, 0.1658, 0.2857],
        [0.2629, 0.2368, 0.1517, 0.1647, 0.1839]], grad_fn=<SelectBackward0>)


In [25]:
out = attn_weights @ V  # (B, T, d_k)
print(out.shape)  # 期望: torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


In [8]:
def single_head_attention(x, W_q, W_k, W_v):
    d_k = x.shape[-1]
    Q = W_q(x)
    K = W_k(x)
    V = W_v(x)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)
    out = attn_weights @ V
    return out, attn_weights

out, attn_weights = single_head_attention(x, W_q, W_k, W_v)
print(out.shape)  # torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


In [9]:
h = 2          # 头数（d_model=8，所以每头 d_k=4）
d_k = d_model // h  # 4

# 多头的线性变换：输入d_model，输出d_model（内部会被拆成h份）
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model, bias=False)  # 最后拼接后的输出投影

Q = W_q(x)  # (B, T, d_model)
K = W_k(x)
V = W_v(x)
print(Q.shape)  # torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


In [10]:
Q = Q.view(B, T, h, d_k).transpose(1, 2)  # (B, h, T, d_k)
K = K.view(B, T, h, d_k).transpose(1, 2)
V = V.view(B, T, h, d_k).transpose(1, 2)
print(Q.shape)  # 期望: torch.Size([2, 2, 5, 4])

torch.Size([2, 2, 5, 4])


In [11]:
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (B, h, T, T)
attn_weights = F.softmax(scores, dim=-1)
out = attn_weights @ V  # (B, h, T, d_k)
print(out.shape)  # 期望: torch.Size([2, 2, 5, 4])

torch.Size([2, 2, 5, 4])


In [12]:
# transpose回来，再拼接
out = out.transpose(1, 2)           # (B, T, h, d_k)
out = out.contiguous().view(B, T, d_model)  # (B, T, d_model)
out = W_o(out)                      # (B, T, d_model)
print(out.shape)  # 期望: torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, d_model = x.shape

        Q = self.W_q(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.h, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        out = attn_weights @ V                              # (B, h, T, d_k)

        out = out.transpose(1, 2).contiguous()             # (B, T, h, d_k)
        out = out.view(B, T, d_model)                      # (B, T, d_model)
        out = self.W_o(out)                                 # (B, T, d_model)
        return out, attn_weights

# 测试
mha = MultiHeadAttention(d_model=8, h=2)
out, attn_weights = mha(x)
print(out.shape)         # torch.Size([2, 5, 8])
print(attn_weights.shape) # torch.Size([2, 2, 5, 5])

torch.Size([2, 5, 8])
torch.Size([2, 2, 5, 5])


In [14]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, d_model = x.shape

        Q = self.W_q(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.h, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        out = attn_weights @ V                              # (B, h, T, d_k)

        out = out.transpose(1, 2).contiguous()             # (B, T, h, d_k)
        out = out.view(B, T, d_model)                      # (B, T, d_model)
        out = self.W_o(out)                                 # (B, T, d_model)
        return out, attn_weights

# 测试
mha = MultiHeadAttention(d_model=8, h=2)
out, attn_weights = mha(x)
print(out.shape)         # torch.Size([2, 5, 8])
print(attn_weights.shape) # torch.Size([2, 2, 5, 5])

torch.Size([2, 5, 8])
torch.Size([2, 2, 5, 5])


In [15]:
# 生成一个 T×T 的causal mask
T = 5
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
print(causal_mask.int())

tensor([[0, 1, 1, 1, 1],
        [0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1],
        [0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0]], dtype=torch.int32)


In [16]:
def single_head_attention_with_mask(x, W_q, W_k, W_v, mask=None):
    d_k = x.shape[-1]
    Q = W_q(x)
    K = W_k(x)
    V = W_v(x)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)
    out = attn_weights @ V
    return out, attn_weights

# 测试
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

out, attn_weights = single_head_attention_with_mask(x, W_q, W_k, W_v, mask=causal_mask)
print(attn_weights[0])  # 看第一个样本，上三角应该全是0

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5355, 0.4645, 0.0000, 0.0000, 0.0000],
        [0.2819, 0.3651, 0.3530, 0.0000, 0.0000],
        [0.1867, 0.2368, 0.3384, 0.2380, 0.0000],
        [0.2341, 0.1530, 0.2385, 0.2206, 0.1539]], grad_fn=<SelectBackward0>)


In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, d_model = x.shape

        Q = self.W_q(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.h, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        out = attn_weights @ V

        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, d_model)
        out = self.W_o(out)
        return out, attn_weights

# 用causal mask测试
mha = MultiHeadAttention(d_model=8, h=2)
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
out, attn_weights = mha(x, mask=causal_mask)

print(out.shape)          # torch.Size([2, 5, 8])
print(attn_weights[0, 0]) # 第一个样本第一个头，上三角应全为0

torch.Size([2, 5, 8])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2911, 0.7089, 0.0000, 0.0000, 0.0000],
        [0.1900, 0.3895, 0.4205, 0.0000, 0.0000],
        [0.2148, 0.2895, 0.2898, 0.2059, 0.0000],
        [0.1187, 0.3348, 0.3599, 0.1175, 0.0692]], grad_fn=<SelectBackward0>)


In [18]:
B, h, T = 2, 2, 5

# 假设第0个样本最后2个位置是pad，第1个样本最后1个位置是pad
padding_mask = torch.tensor([
    [False, False, False, True, True],
    [False, False, False, False, True]
])  # (B, T) = (2, 5)

# reshape让它能broadcast到scores
padding_mask = padding_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T) = (2, 1, 1, 5)
print(padding_mask.shape)

# scores (B, h, T, T) = (2, 2, 5, 5)
scores = torch.randn(B, h, T, T)
scores.masked_fill(padding_mask, float('-inf'))
# 每个样本的pad列全部变成-inf

torch.Size([2, 1, 1, 5])


tensor([[[[ 0.5872,  0.1818,  0.0644,    -inf,    -inf],
          [ 1.0834,  0.3723,  0.3502,    -inf,    -inf],
          [ 0.8192,  1.3987,  0.0504,    -inf,    -inf],
          [ 0.3530,  0.0976,  2.1015,    -inf,    -inf],
          [-1.5519,  0.2253,  0.9839,    -inf,    -inf]],

         [[-1.7446,  0.9857,  0.9841,    -inf,    -inf],
          [-0.6034,  1.0913,  1.7378,    -inf,    -inf],
          [ 0.1054,  1.6494, -1.0048,    -inf,    -inf],
          [ 0.9139,  0.2343, -0.4211,    -inf,    -inf],
          [-0.4800,  1.8272, -0.0340,    -inf,    -inf]]],


        [[[ 1.3651,  0.7741,  1.1110, -0.1038,    -inf],
          [ 1.2456, -1.4215, -0.1024, -0.5055,    -inf],
          [ 0.9081, -0.2124, -0.1227, -1.9202,    -inf],
          [ 1.4242,  0.2759,  0.4368, -0.3205,    -inf],
          [ 0.7245,  0.8863, -1.4887,  1.2498,    -inf]],

         [[-0.4380, -0.1408,  0.1651,  0.9201,    -inf],
          [-0.1504,  0.8704, -0.1341, -0.9429,    -inf],
          [-2.1048,  1.